## Google Street View data retrival

Load configuration, define reusable functions, then run the target range.

### Setup

In [1]:
import csv
import importlib
import json
import math
import os
import re
import sys
import time
import requests
from pathlib import Path

def load_env_file(filename=".env"):
    """Load KEY=VALUE pairs from the nearest .env file into os.environ."""
    for base in [Path.cwd(), *Path.cwd().parents]:
        env_path = base / filename
        if not env_path.is_file():
            continue

        for raw_line in env_path.read_text().splitlines():
            line = raw_line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue

            key, value = line.split("=", 1)
            key = key.strip()
            value = value.strip().strip("\"'")
            os.environ.setdefault(key, value)

        return env_path

    return None

def find_project_root():
    """Find the repository root by locating data/cleaned_data.csv."""
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "data" / "cleaned_data.csv").is_file():
            return base

    raise FileNotFoundError("Could not locate data/cleaned_data.csv from the current notebook directory.")

def safe_entry_id(value):
    """Make sure filenames are stable and filesystem-safe."""
    cleaned = re.sub(r"[^A-Za-z0-9_-]+", "_", str(value).strip())
    return cleaned.strip("_") or "entry"

PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import gsv_pipeline
gsv_pipeline = importlib.reload(gsv_pipeline)

DATA_PATH = PROJECT_ROOT / "data" / "cleaned_data.csv"
OUT_DIR = PROJECT_ROOT / "notebook" / "gsv_out" / "cleaned_data_first50"

RUN_CONFIG = {
    "start": 1,
    "end": 50,
    "target_date_field": "create_date",
    "candidate_radius": 25,
    "max_distance_meters": 20,
    "max_baseline_shift_meters": 10,
    "require_baseline": True,
    "allow_distance_fallback": False,
}

ENV_PATH = load_env_file()
API_KEY = os.environ.get("GSV_API_KEY")
if not API_KEY:
    raise RuntimeError(
        "Missing GSV_API_KEY. Copy .env.example to .env, use a key with both Street View Static API and Geocoding API enabled, and rerun this cell."
    )

{
    "env_path": str(ENV_PATH) if ENV_PATH else None,
    "data_path": str(DATA_PATH),
    "output_dir": str(OUT_DIR),
    "run_config": RUN_CONFIG,
}

{'env_path': '/Volumes/E/Leo/Study/CS Capstone/satellite-data/.env',
 'data_path': '/Volumes/E/Leo/Study/CS Capstone/satellite-data/data/cleaned_data.csv',
 'output_dir': '/Volumes/E/Leo/Study/CS Capstone/satellite-data/notebook/gsv_out/cleaned_data_first50'}

### Street View helpers

In [2]:

# ===== 1) Utility: Geographic helpers =====
def bearing_deg(lat1, lon1, lat2, lon2):
    """Return heading in degrees from point1 (camera) to point2 (house)."""
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dlon = math.radians(lon2 - lon1)
    y = math.sin(dlon) * math.cos(phi2)
    x = math.cos(phi1) * math.sin(phi2) - math.sin(phi1) * math.cos(phi2) * math.cos(dlon)
    brng = math.degrees(math.atan2(y, x))
    return (brng + 360) % 360

def haversine_meters(lat1, lon1, lat2, lon2):
    """Return great-circle distance in meters between two lat/lng pairs."""
    radius_m = 6371000
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = (
        math.sin(dphi / 2) ** 2
        + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2) ** 2
    )
    return 2 * radius_m * math.atan2(math.sqrt(a), math.sqrt(1 - a))

def format_lat_lng(lat, lon):
    """Format a lat/lng pair the way Google web services expect."""
    return f"{lat:.8f},{lon:.8f}"

# ===== 2) Address cleanup + Geocoding API =====
def normalize_house_address(address, default_state="PA"):
    """Normalize CSV address strings so geocoding is more likely to hit the exact parcel."""
    address = (address or "").strip()
    if not address:
        return None
    if address.lower() == "no primary address specified":
        return None

    normalized = re.sub(r"\s+", " ", address)
    normalized = re.sub(r"\s+,", ",", normalized)

    if re.search(r",\s*[A-Z]{2}\s+\d{5}(?:-\d{4})?$", normalized, re.IGNORECASE):
        return normalized

    zip_match = re.search(r",\s*(\d{5}(?:-\d{4})?)$", normalized)
    if zip_match:
        return re.sub(
            r",\s*(\d{5}(?:-\d{4})?)$",
            f", {default_state} \\1",
            normalized,
        )

    if not re.search(r",\s*[A-Z]{2}(?:\s+\d{5}(?:-\d{4})?)?$", normalized, re.IGNORECASE):
        return f"{normalized}, {default_state}"

    return normalized

def geocode_address(address, api_key, region="us"):
    """Convert an address string into a geocoded point."""
    url = "https://maps.googleapis.com/maps/api/geocode/json"
    params = {
        "address": address,
        "key": api_key,
    }
    if region:
        params["region"] = region

    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()
    return response.json()

def summarize_geocode_result(geocode_response):
    """Keep the most relevant geocode fields for downstream decision-making."""
    if geocode_response.get("status") != "OK" or not geocode_response.get("results"):
        return None

    result = geocode_response["results"][0]
    geometry = result.get("geometry", {})
    location = geometry.get("location", {})
    return {
        "formatted_address": result.get("formatted_address"),
        "place_id": result.get("place_id"),
        "partial_match": bool(result.get("partial_match")),
        "location_type": geometry.get("location_type"),
        "lat": location.get("lat"),
        "lon": location.get("lng"),
    }

def choose_house_target(address, api_key, fallback_lat=None, fallback_lon=None, region="us", default_state="PA"):
    """
    At this stage, prefer the street address whenever one exists.

    Principle:
    - if the CSV has a usable street address, use that path end-to-end;
    - only fall back to dataset coordinates when the address is missing or says
      'No primary address specified'.
    """
    normalized_address = normalize_house_address(address, default_state=default_state)
    geocode_response = None
    geocode_result = None
    target_lat = None
    target_lon = None
    target_source = None

    if normalized_address:
        geocode_response = geocode_address(normalized_address, api_key, region=region)
        geocode_result = summarize_geocode_result(geocode_response)

        if geocode_result and geocode_result.get("lat") is not None and geocode_result.get("lon") is not None:
            target_lat = geocode_result["lat"]
            target_lon = geocode_result["lon"]
            target_source = "geocode_address"

    if (target_lat is None or target_lon is None) and not normalized_address:
        if fallback_lat is not None and fallback_lon is not None:
            target_lat = fallback_lat
            target_lon = fallback_lon
            target_source = "dataset_coordinates"

    return {
        "normalized_address": normalized_address,
        "geocode_status": geocode_response.get("status") if geocode_response else None,
        "geocode_result": geocode_result,
        "geocode_response": geocode_response,
        "target_lat": target_lat,
        "target_lon": target_lon,
        "target_source": target_source,
    }

# ===== 3) Street View metadata lookup =====
def get_sv_metadata(location, api_key, radius=None):
    """Query Street View metadata using either an address string or a lat/lng pair."""
    url = "https://maps.googleapis.com/maps/api/streetview/metadata"
    if isinstance(location, str):
        location_value = location
    else:
        location_value = format_lat_lng(location[0], location[1])

    params = {
        "location": location_value,
        "key": api_key,
    }
    if radius is not None and not isinstance(location, str):
        params["radius"] = radius

    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()
    return response.json()

def find_best_pano(address, target_lat, target_lon, api_key, radius=50):
    """
    Prefer address lookup whenever the CSV has a real street address.
    Only fall back to coordinate lookup when there is no usable address.
    """
    attempts = []
    if address:
        address_meta = get_sv_metadata(address, api_key)
        attempts.append({
            "lookup_type": "address",
            "lookup_value": address,
            "status": address_meta.get("status"),
        })
        return address_meta, attempts, "address"

    if target_lat is not None and target_lon is not None:
        coord_meta = get_sv_metadata((target_lat, target_lon), api_key, radius=radius)
        attempts.append({
            "lookup_type": "coordinates",
            "lookup_value": format_lat_lng(target_lat, target_lon),
            "status": coord_meta.get("status"),
            "radius": radius,
        })
        if coord_meta.get("status") == "OK":
            return coord_meta, attempts, "coordinates"

        return coord_meta, attempts, "coordinates"

    return {"status": "INVALID_REQUEST", "error_message": "Missing both address and coordinates."}, attempts, None

# ===== 4) Download Street View image =====
def download_sv_image(
    pano_id, cam_lat, cam_lon, heading, api_key,
    out_path, size="640x640", fov=80, pitch=0, return_error_code=True
):
    """
    Download a Street View image for a specific pano so the image request does not snap again.
    """
    url = "https://maps.googleapis.com/maps/api/streetview"
    params = {
        "size": size,
        "fov": fov,
        "pitch": pitch,
        "heading": heading,
        "key": api_key,
    }
    if return_error_code:
        params["return_error_code"] = "true"
    if pano_id:
        params["pano"] = pano_id
    else:
        params["location"] = format_lat_lng(cam_lat, cam_lon)

    response = requests.get(url, params=params, timeout=60)
    response.raise_for_status()

    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_bytes(response.content)
    return str(out_path)

def write_sv_metadata(meta_path, payload):
    """Save Street View metadata next to the downloaded image."""
    meta_path = Path(meta_path)
    meta_path.parent.mkdir(parents=True, exist_ok=True)
    meta_path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")
    return str(meta_path)

# ===== 5) Process a single entry: address -> pano -> heading -> image =====
def fetch_front_gsv_for_house(
    address, api_key,
    house_lat=None, house_lon=None,
    out_dir="gsv_out", entry_id="sample",
    radius=50, size="640x640", fov=80, pitch=0,
    region="us", default_state="PA"
):
    """
    Fetch a front-facing Street View image using address-first pano lookup.

    Current stage policy:
    - if the row has a real street address, use address-driven lookup;
    - if the row says 'No primary address specified', fall back to dataset coordinates.
    """
    target = choose_house_target(
        address=address,
        api_key=api_key,
        fallback_lat=house_lat,
        fallback_lon=house_lon,
        region=region,
        default_state=default_state,
    )

    target_lat = target["target_lat"]
    target_lon = target["target_lon"]
    if target_lat is None or target_lon is None:
        return {
            "ok": False,
            "status": "INVALID_REQUEST",
            "message": "Missing both a geocodable address and fallback coordinates.",
            "target": target,
        }

    meta, lookup_attempts, lookup_source = find_best_pano(
        address=target["normalized_address"],
        target_lat=target_lat,
        target_lon=target_lon,
        api_key=api_key,
        radius=radius,
    )

    status = meta.get("status")
    if status != "OK":
        return {
            "ok": False,
            "status": status,
            "message": meta.get("error_message"),
            "lookup_attempts": lookup_attempts,
            "target": target,
            "meta": meta,
        }

    cam_loc = meta["location"]
    cam_lat, cam_lon = cam_loc["lat"], cam_loc["lng"]
    heading = bearing_deg(cam_lat, cam_lon, target_lat, target_lon)
    distance_meters = haversine_meters(cam_lat, cam_lon, target_lat, target_lon)

    out_path = Path(out_dir) / f"{entry_id}_front.jpg"
    saved_path = download_sv_image(
        pano_id=meta.get("pano_id"),
        cam_lat=cam_lat,
        cam_lon=cam_lon,
        heading=heading,
        api_key=api_key,
        out_path=out_path,
        size=size,
        fov=fov,
        pitch=pitch,
    )
    meta_path = out_path.with_suffix(".meta.json")
    saved_meta_path = write_sv_metadata(
        meta_path,
        {
            "entry_id": entry_id,
            "input_address": address,
            "normalized_address": target["normalized_address"],
            "fallback_house_lat": house_lat,
            "fallback_house_lon": house_lon,
            "target_lat": target_lat,
            "target_lon": target_lon,
            "target_source": target["target_source"],
            "lookup_source": lookup_source,
            "lookup_attempts": lookup_attempts,
            "cam_lat": cam_lat,
            "cam_lon": cam_lon,
            "heading": heading,
            "distance_meters": distance_meters,
            "image_path": saved_path,
            "geocode_status": target["geocode_status"],
            "geocode_result": target["geocode_result"],
            "geocode_response": target["geocode_response"],
            "gsv_metadata": meta,
        },
    )

    return {
        "ok": True,
        "status": status,
        "entry_id": entry_id,
        "input_address": address,
        "normalized_address": target["normalized_address"],
        "house_lat": target_lat,
        "house_lon": target_lon,
        "target_source": target["target_source"],
        "lookup_source": lookup_source,
        "lookup_attempts": lookup_attempts,
        "cam_lat": cam_lat,
        "cam_lon": cam_lon,
        "distance_meters": distance_meters,
        "heading": heading,
        "pano_id": meta.get("pano_id"),
        "date": meta.get("date"),
        "image_path": saved_path,
        "meta_path": saved_meta_path,
        "target": target,
        "meta": meta,
    }


### Batch loading

In [3]:

def load_house_entries(csv_path, start=1, end=50):
    """Load rows in the inclusive range [start, end] from cleaned_data.csv."""
    if start < 1:
        raise ValueError("start must be >= 1")
    if end < start:
        raise ValueError("end must be >= start")

    entries = []
    with csv_path.open(newline="", encoding="utf-8") as handle:
        reader = csv.DictReader(handle)
        for index, row in enumerate(reader, start=1):
            if index < start:
                continue
            if index > end:
                break

            address = (row.get("address") or "").strip()
            usable_address = bool(address) and address.lower() != "no primary address specified"
            lat_raw = (row.get("latitude") or "").strip()
            lon_raw = (row.get("longitude") or "").strip()
            has_coords = bool(lat_raw and lon_raw)

            if not usable_address and not has_coords:
                entries.append({
                    "index": index,
                    "row": row,
                    "skip_reason": "missing_address_and_coordinates",
                })
                continue

            entry = {
                "index": index,
                "row": row,
                "address": address or None,
                "entry_id": f"{index:03d}_{safe_entry_id(row.get('_id') or row.get('parcel_id') or 'entry')}",
            }
            if has_coords:
                entry["lat"] = float(lat_raw)
                entry["lon"] = float(lon_raw)

            entries.append(entry)

    return entries

def fetch_entries(
    csv_path, api_key, start=1, end=50, out_dir="gsv_out", pause_seconds=0.1,
    region="us", default_state="PA"
):
    """Download front-facing Street View images for cleaned-data rows in [start, end]."""
    results = []
    for entry in load_house_entries(csv_path, start=start, end=end):
        row = entry["row"]
        result = {
            "index": entry["index"],
            "_id": row.get("_id"),
            "parcel_id": row.get("parcel_id"),
            "address": row.get("address"),
        }

        if entry.get("skip_reason"):
            result.update({
                "ok": False,
                "status": "SKIPPED",
                "message": entry["skip_reason"],
            })
            results.append(result)
            continue

        try:
            fetch_result = fetch_front_gsv_for_house(
                address=entry.get("address"),
                house_lat=entry.get("lat"),
                house_lon=entry.get("lon"),
                api_key=api_key,
                out_dir=out_dir,
                entry_id=entry["entry_id"],
                region=region,
                default_state=default_state,
            )
            result.update(fetch_result)
        except requests.HTTPError as exc:
            result.update({
                "ok": False,
                "status": "HTTP_ERROR",
                "message": str(exc),
            })
        except requests.RequestException as exc:
            result.update({
                "ok": False,
                "status": "REQUEST_ERROR",
                "message": str(exc),
            })
        except Exception as exc:
            result.update({
                "ok": False,
                "status": type(exc).__name__,
                "message": str(exc),
            })

        results.append(result)
        if pause_seconds:
            time.sleep(pause_seconds)

    return results


### Run a range

In [4]:
results = gsv_pipeline.fetch_entries(
    csv_path=DATA_PATH,
    api_key=API_KEY,
    start=RUN_CONFIG["start"],
    end=RUN_CONFIG["end"],
    out_dir=OUT_DIR,
    target_date_field=RUN_CONFIG["target_date_field"],
    candidate_radius=RUN_CONFIG["candidate_radius"],
    max_distance_meters=RUN_CONFIG["max_distance_meters"],
    max_baseline_shift_meters=RUN_CONFIG["max_baseline_shift_meters"],
    require_baseline=RUN_CONFIG["require_baseline"],
    allow_distance_fallback=RUN_CONFIG["allow_distance_fallback"],
)

downloaded = [item for item in results if item.get("ok")]
failed = [item for item in results if not item.get("ok")]

{
    "requested_range": [RUN_CONFIG["start"], RUN_CONFIG["end"]],
    "requested_entries": RUN_CONFIG["end"] - RUN_CONFIG["start"] + 1,
    "downloaded_images": len(downloaded),
    "failed_entries": len(failed),
    "sample_results": results[:5],
}

{'requested_range': [1, 50],
 'requested_entries': 50,
 'downloaded_images': 49,
 'failed_entries': 1,
 'sample_results': [{'index': 1,
   '_id': '53193',
   'parcel_id': '0004J00093000000',
   'address': '511 GRACE ST, Pittsburgh, 15211',
   'ok': True,
   'status': 'OK',
   'entry_id': '001_53193',
   'input_address': '511 GRACE ST, Pittsburgh, 15211',
   'normalized_address': '511 GRACE ST, Pittsburgh, PA 15211',
   'house_lat': 40.4277862,
   'house_lon': -80.0147058,
   'target_source': 'geocode_address',
   'lookup_source': 'address',
   'lookup_attempts': [{'lookup_type': 'address',
     'lookup_value': '511 GRACE ST, Pittsburgh, PA 15211',
     'status': 'OK'}],
   'cam_lat': 40.42783129916447,
   'cam_lon': -80.01483704793196,
   'distance_meters': 12.188784376652663,
   'heading': 114.29448271778347,
   'pano_id': 'HG3C971dmCW71cbwegOd5w',
   'date': '2024-11',
   'image_path': '/Volumes/E/Leo/Study/CS Capstone/satellite-data/notebook/gsv_out/cleaned_data_first50/001_53193_fr